# Phase 16J — controlled CUDA smoke tests

This notebook uses the existing Phase 16I trainer, four YAML configurations, and frozen Phase 16H datasets. Run cells manually in order on a Google Colab **NVIDIA GPU** runtime (T4 16 GB or better preferred). GPU availability, memory, and compatibility must pass the preflight. Installing packages or executing smoke tests may consume a Colab session; no cell requests a paid service. The two smoke switches default to `False`, so **Run all does not start training**. Full training commands appear only in the documentation.

Build the upload archive locally using `python scripts/bundle_colab.py --phase16i-frozen --output outputs/phase16j-frozen-upload.tar.gz`. It contains only allowlisted project code and the frozen training inputs, never credentials or model weights. See `docs/training/phase16j_colab_setup.md`.

## 1. GPU verification

In Colab, select **Runtime → Change runtime type → GPU** before running. This checks the assigned device and driver before installation.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

smi = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,memory.free,driver_version",
        "--format=csv,noheader,nounits",
    ],
    capture_output=True,
    text=True,
)
print(smi.stdout or smi.stderr)
if smi.returncode:
    raise RuntimeError("No NVIDIA GPU runtime is visible. Stop here and select a GPU runtime.")

## 2. Repository setup

Upload the allowlisted archive built on your Mac. Extraction refuses links, unexpected paths, and an existing project directory. If using a private Git clone instead, copy the five frozen files at their exact relative paths before continuing; the frozen files are normally untracked.

In [ ]:
import tarfile

from google.colab import files

PROJECT = Path("/content/multilingual-llm-reasoning")
if PROJECT.exists():
    raise FileExistsError(f"{PROJECT} already exists; inspect it rather than overwriting")
uploaded = files.upload()
archives = [Path(name) for name in uploaded if name.endswith(".tar.gz")]
if len(archives) != 1:
    raise ValueError("Upload exactly one archive built with --phase16i-frozen")
with tarfile.open(archives[0], "r:gz") as archive:
    members = archive.getmembers()
    for member in members:
        parts = Path(member.name).parts
        if (
            not parts
            or parts[0] != PROJECT.name
            or ".." in parts
            or not (member.isfile() or member.isdir())
            or member.issym()
            or member.islnk()
        ):
            raise ValueError(f"Unsafe or unexpected archive member: {member.name}")
    archive.extractall(PROJECT.parent, filter="data")
assert (PROJECT / "configs/phase16i_english_smoke.yaml").is_file()
assert (PROJECT / "outputs/phase16i-training-preparation-v1/sft_training_manifest.json").is_file()
print("Project:", PROJECT)

## 3. Locked dependency installation

The existing installer creates an isolated Python 3.11.14 environment; it does not alter the notebook interpreter. Its hash-locked requirements include CUDA PyTorch, Transformers, PEFT, Accelerate, TRL, datasets, and bitsandbytes. Installation downloads CUDA packages, but does not download model weights.

In [ ]:
subprocess.run(["bash", "scripts/setup_colab.sh"], cwd=PROJECT, check=True)
PYTHON = PROJECT / ".venv-colab/bin/python"
ENV = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"}
subprocess.run(
    [
        str(PYTHON),
        "-c",
        "import torch, transformers, peft, accelerate, trl, bitsandbytes; "
        "print('torch', torch.__version__, 'CUDA', torch.version.cuda, "
        "'available', torch.cuda.is_available()); "
        "print('transformers', transformers.__version__, 'peft', peft.__version__, "
        "'accelerate', accelerate.__version__, 'trl', trl.__version__, "
        "'bitsandbytes', bitsandbytes.__version__)",
    ],
    cwd=PROJECT,
    env=ENV,
    check=True,
)
print("Frozen environment:", PROJECT / ".venv-colab/environment.freeze.txt")

## 4. Frozen dataset integrity and model revision

The exact SHA-256 values below are the Phase 16H freeze. The model revision is checked against both YAML configs and the freeze manifest. A lightweight Hub metadata query checks access to that exact revision without downloading weights; the trainer later loads that revision.

In [ ]:
import hashlib

FREEZE = PROJECT / "outputs/phase16h-approved-sft-freeze-v1"
EXPECTED = {
    "english_only_sft_final.jsonl": (
        "6e59b5344db55eb84f87c9dc9059587fbd00dae0edcea967a76d37705b8ed881"
    ),
    "multilingual_sft_final.jsonl": (
        "9b5c813da240229a31a7e57d1bc8a8e4cb9e8a410a837c38b01db23202df2021"
    ),
}
for name, digest in EXPECTED.items():
    actual = hashlib.sha256((FREEZE / name).read_bytes()).hexdigest()
    print(name, actual)
    assert actual == digest, f"Frozen dataset changed: {name}"
freeze_manifest = json.loads((FREEZE / "dataset_freeze_manifest.json").read_text())
assert freeze_manifest["training_ready"] is True
assert all(
    freeze_manifest["dataset_files_sha256"][name] == digest for name, digest in EXPECTED.items()
)
REVISION = "aa8e72537993ba99e69dfaafa59ed015b17504d1"
for name in ("english_smoke", "multilingual_smoke", "english_lora", "multilingual_lora"):
    parsed = subprocess.run(
        [
            str(PYTHON),
            "-c",
            "import json,sys,yaml; print(json.dumps(yaml.safe_load(open(sys.argv[1]).read())))",
            str(PROJECT / f"configs/phase16i_{name}.yaml"),
        ],
        cwd=PROJECT,
        env=ENV,
        check=True,
        capture_output=True,
        text=True,
    )
    config = json.loads(parsed.stdout)
    assert config["model"] == "Qwen/Qwen2.5-3B-Instruct"
    assert config["revision"] == REVISION
assert freeze_manifest["model_revision"] == freeze_manifest["tokenizer_revision"] == REVISION
subprocess.run(
    [
        str(PYTHON),
        "-c",
        "from huggingface_hub import HfApi; "
        "import sys; "
        "r = HfApi().model_info(sys.argv[1], revision=sys.argv[2]); "
        "print('Resolved model SHA:', r.sha); "
        "assert r.sha == sys.argv[2]",
        "Qwen/Qwen2.5-3B-Instruct",
        REVISION,
    ],
    cwd=PROJECT,
    env=ENV,
    check=True,
)

## 5. CUDA preflight

This uses the existing paired-config preflight. It checks installed versions, CUDA device/VRAM, NF4 kernel, disk space, the approved freeze and data split integrity. It creates a report without starting training. **Stop if status is blocked**; do not lower the threshold for just one condition.

In [ ]:
PREFLIGHT = PROJECT / "outputs/phase16j-colab/cuda_preflight.json"
subprocess.run(
    [str(PYTHON), "scripts/phase16i_cuda_preflight.py", "--output", str(PREFLIGHT)],
    cwd=PROJECT,
    env=ENV,
    check=True,
)
preflight = json.loads(PREFLIGHT.read_text())
print(
    json.dumps(
        {
            key: preflight[key]
            for key in (
                "status",
                "model",
                "model_revision",
                "qlora_kernel_probe",
                "disk_free_gib",
                "blockers",
            )
        },
        indent=2,
    )
)
if preflight["status"] != "ready" or preflight["qlora_kernel_probe"] != "passed":
    raise RuntimeError("CUDA preflight blocked training. Read the report and stop.")

## 6. English-only 20-step smoke test

Set `RUN_ENGLISH_SMOKE = True` only after reviewing preflight and your Colab runtime terms. The existing CLI checks response-only loss and runs **20 optimizer steps**, with checkpoint/adapter verification. It can download the pinned Qwen weights on first execution. Outputs remain separate from final training.

In [ ]:
RUN_ENGLISH_SMOKE = False
ENGLISH_RUN = PROJECT / "outputs/training/phase16i_english_smoke/smoke"
if RUN_ENGLISH_SMOKE:
    assert preflight["status"] == "ready"
    subprocess.run(
        [
            str(PYTHON),
            "-m",
            "multilingual_llm_reasoning",
            "train",
            "--config",
            "configs/phase16i_english_smoke.yaml",
            "--mode",
            "smoke",
            "--confirm-gpu-training",
        ],
        cwd=PROJECT,
        env=ENV,
        check=True,
    )
else:
    print("English smoke skipped; set RUN_ENGLISH_SMOKE=True to execute explicitly.")

## 7. Multilingual 20-step smoke test

Set `RUN_MULTILINGUAL_SMOKE = True` only after the English smoke is complete and you decide to run the second condition. Both configs have the same optimizer, seed, sequence length, batch/accumulation, LoRA settings, QLoRA precision, and 20 optimizer steps; the training dataset is the intended difference.

In [ ]:
RUN_MULTILINGUAL_SMOKE = False
MULTILINGUAL_RUN = PROJECT / "outputs/training/phase16i_multilingual_smoke/smoke"
if RUN_MULTILINGUAL_SMOKE:
    assert preflight["status"] == "ready"
    subprocess.run(
        [
            str(PYTHON),
            "-m",
            "multilingual_llm_reasoning",
            "train",
            "--config",
            "configs/phase16i_multilingual_smoke.yaml",
            "--mode",
            "smoke",
            "--confirm-gpu-training",
        ],
        cwd=PROJECT,
        env=ENV,
        check=True,
    )
else:
    print("Multilingual smoke skipped; set RUN_MULTILINGUAL_SMOKE=True to execute explicitly.")

## 8. Compare smoke-test results

Only actual completed `run.json` and `metrics.json` files count. The trainer checks finite losses, adapter updates, checkpoint reload, and peak CUDA memory. This cell rechecks those outputs and the frozen dataset hashes; it prints pending if either run has not executed.

In [ ]:
import math

rows = []
for condition, directory in (("english", ENGLISH_RUN), ("multilingual", MULTILINGUAL_RUN)):
    run_path, metrics_path = directory / "run.json", directory / "metrics.json"
    if not run_path.is_file() or not metrics_path.is_file():
        print(condition, "PENDING — no completed CUDA smoke artifact")
        continue
    run, metrics = json.loads(run_path.read_text()), json.loads(metrics_path.read_text())
    assert run["status"] == "completed" and metrics["global_step"] == 20
    assert metrics["smoke_checkpoint_load_verified"] is True
    assert metrics["smoke_optimizer_update_verified"] is True
    assert (directory / "checkpoint-20/adapter_model.safetensors").is_file()
    for key, value in metrics.items():
        if "loss" in key:
            assert math.isfinite(float(value)), (condition, key, value)
    rows.append(
        {
            "condition": condition,
            "optimizer_steps": 20,
            "train_loss": metrics.get("train_loss"),
            "eval_loss": metrics.get("eval_loss"),
            "runtime_seconds": metrics["runtime_seconds"],
            "peak_allocated_gib": metrics["peak_allocated_gib"],
            "peak_reserved_gib": metrics["peak_reserved_gib"],
        }
    )
for name, digest in EXPECTED.items():
    assert hashlib.sha256((FREEZE / name).read_bytes()).hexdigest() == digest
print(json.dumps(rows, indent=2))
if len(rows) != 2:
    print("Both conditions must complete before comparing their measured results.")

## 9. Persist checkpoints to Google Drive

Optional and deliberately disabled. Mount Drive and copy only after both smoke runs have passed the checks above. The copy includes checkpoint state needed for exact resume, logs, metrics, environment freeze, and preflight. It will never overwrite an existing destination. Drive storage and data-sharing policy are your responsibility.

In [ ]:
PERSIST_TO_DRIVE = False
if PERSIST_TO_DRIVE:
    if len(rows) != 2:
        raise RuntimeError("Complete and verify both smoke tests before persistence")
    import shutil

    from google.colab import drive

    drive.mount("/content/drive")
    DEST = Path("/content/drive/MyDrive/multilingual-llm-reasoning/phase16j-smoke-v1")
    if DEST.exists():
        raise FileExistsError(f"Choose a new Drive destination; refusing overwrite: {DEST}")
    DEST.mkdir(parents=True)
    shutil.copytree(ENGLISH_RUN, DEST / "english_smoke")
    shutil.copytree(MULTILINGUAL_RUN, DEST / "multilingual_smoke")
    shutil.copy2(PREFLIGHT, DEST / "cuda_preflight.json")
    shutil.copy2(PROJECT / ".venv-colab/environment.freeze.txt", DEST / "environment.freeze.txt")
    for name in ("english_smoke", "multilingual_smoke"):
        shutil.copy2(PROJECT / f"configs/phase16i_{name}.yaml", DEST / f"phase16i_{name}.yaml")
    print("Persisted smoke artifacts:", DEST)
else:
    print("Drive persistence skipped; set PERSIST_TO_DRIVE=True only after verification.")

## 10. Eventual full training — instructions only

**Do not run full training in this Phase 16J notebook.** After both smoke tests pass and a separate decision to train, use the existing full-run configs `configs/phase16i_english_lora.yaml` and `configs/phase16i_multilingual_lora.yaml` in independent sessions. The CLI commands and resume instructions are in `docs/training/phase16j_colab_setup.md`. The full-run configs target about 75 optimizer steps per condition; verify the exact dry-run count on CUDA before launch. Preserve complete checkpoint directories on Drive if you need resume. No full-training command is executed here.